In [1]:
#pip install grakel

In [4]:
#pip install --upgrade pip

In [2]:
#pip install numpy==1.23.5

In [1]:
#pip install POT

In [1]:
import numpy as np
import os
import logging
from grakel import datasets
from data_loader import load_local_data
from nested_cv_fgw import nested_fgw
from nested_cv_grakel import nested_cv
from utils import setup_logger, create_log_dir
import itertools
from gwgraphs import *

In [2]:
def evaluate_graph_kernels_nested_cv(dataset_name,
                                     G,
                                     y,
                                     log_dir,
                                     n_folds=10,
                                     n_inner=10,
                                     random_state=42):
    """
    Evaluate graph kernel methods (hopper and propa) using nested cross-validation.

    Parameters
    ----------
    dataset_name : str
        Name of the dataset being evaluated
    G : array
        Graph data in grakel format
    y : array
        Labels
    log_dir : str
        Directory for log files
    n_folds : int
        Number of outer cross-validation folds
    n_inner : int
        Number of inner cross-validation folds
    random_state : int
        Random state for reproducibility
    """
    results = {}

    # Define parameters for each method
    methods = {
        'hopper': [{
            'C':
            list(np.logspace(-5, 5, 15)),
            'kernel_params': [[{
                "name": "graph_hopper",
                "kernel_type": 'linear'
            }]],
            'normalize': [True]
        }],
        'propa': [{
            'C':
            list(np.logspace(-5, 5, 15)),
            'kernel_params': [[{
                "name": "propagation",
                "t_max": t,
                "w": 1e-4,
                "M": "TV"
            }] for t in [1, 3, 5, 8, 10, 15, 20]],
            'normalize': [True]
        }]
    }

    print(
        f"\nEvaluating graph kernel methods on {dataset_name} dataset using nested CV..."
    )

    for method_name, param_grid in methods.items():
        print(f"\n--- Evaluating {method_name} method with nested CV ---")

        # Setup logger for this method
        logger_name = f'{method_name}_{dataset_name}'
        logger = setup_logger(logger_name,
                              f'{log_dir}/{logger_name}_nested_cv.log')

        try:
            # Run nested cross-validation
            # Ensure y is a 1D array
            y_flat = np.array(y).ravel()
            nested_cv(G,
                      y_flat,
                      param_grid,
                      logger,
                      n_iter=n_folds,
                      n_inner=n_inner,
                      verbose=1)

            results[method_name] = "Completed - check log file for results"
            print(f"  {method_name} nested CV completed - results in log file")

        except Exception as e:
            print(f"  Error in {method_name} nested CV: {e}")
            results[method_name] = f"Error: {e}"

    return results


def evaluate_fgw_nested_cv(dataset_name,
                           X,
                           y,
                           log_dir,
                           n_folds=10,
                           n_inner=10,
                           random_state=42):
    """
    Evaluate FGW method using nested cross-validation.

    Parameters
    ----------
    dataset_name : str
        Name of the dataset being evaluated
    X : array
        Graph objects
    y : array
        Labels
    log_dir : str
        Directory for log files
    n_folds : int
        Number of outer cross-validation folds
    n_inner : int
        Number of inner cross-validation folds
    random_state : int
        Random state for reproducibility
    """
    print(
        f"\n--- Evaluating FGW method on {dataset_name} dataset with nested CV ---"
    )

    # Setup logger for FGW
    logger_name = f'fgw_{dataset_name}'
    logger = setup_logger(logger_name,
                          f'{log_dir}/{logger_name}_nested_cv.log')

    # Parameter grid for FGW nested CV
    # Using automatic CV alpha grid
    nb = 15
    N = int(15 / 3)
    a = np.logspace(-6, -1, N)
    c = 1 - a
    b = np.array(
        list(set(np.linspace(a[0], c[0], N)).difference(set((a[0], c[0])))))
    alphalist = np.concatenate((a, b, c))
    alpha_list = list(np.sort(np.append([0, 1], alphalist)))

    tuned_parameters = [{
        'alpha': alpha_list,
        'C': list(np.logspace(-4, 4, 15)),
        'gamma': list([2**k for k in np.linspace(-10, 10, 15)]),
        'features_metric': ['sqeuclidean'],
        'method': ['shortest_path'],
        'wl': [0],
        'amijo': [True]
    }]

    try:
        # Run nested cross-validation
        # Ensure y is a 1D array
        y_flat = np.array(y).ravel()
        nested_fgw(X,
                   y_flat,
                   tuned_parameters,
                   dataset_name,
                   logger,
                   path=None,
                   n_inner=n_inner,
                   n_iter=n_folds,
                   verbose=1)

        print(f"  FGW nested CV completed - results in log file")
        return "Completed - check log file for results"

    except Exception as e:
        print(f"  Error in FGW nested CV: {e}")
        return f"Error: {e}"


def load_dataset_from_files(dataset_name):
    """
    Load datasets from local files.

    Parameters
    ----------
    dataset_name : str
        Either 'bzr' or 'cox2'

    Returns
    -------
    tuple : (graphs, labels) for both grakel format and local format
    """
    print(f"Loading {dataset_name} dataset...")

    # Load using local data loader for FGW methods
    X_local, y_local = load_local_data('.',
                                       dataset_name.lower(),
                                       attributes=True)

    # For graph kernel methods, we'll use the same data
    # Convert our Graph objects to a format compatible with grakel
    G_grakel = []
    for graph in X_local:
        # Convert to networkx format that grakel expects
        nx_graph = graph.nx_graph
        # Create adjacency matrix representation
        nodes = sorted(list(nx_graph.nodes()))  # Sort for consistent ordering
        node_mapping = {node: i for i, node in enumerate(nodes)}

        # Convert edges to use integer indices
        edges = []
        for u, v in nx_graph.edges():
            if u in node_mapping and v in node_mapping:
                u_idx = node_mapping[u]
                v_idx = node_mapping[v]
                edges.append((u_idx, v_idx))

        # Get node attributes if they exist and ensure consistent format
        node_attrs = {}
        for i, node in enumerate(nodes):
            if node in nx_graph.nodes():
                attrs = nx_graph.nodes[node]
                if 'attr_name' in attrs:
                    attr_val = attrs['attr_name']
                    # Ensure attribute is a scalar value
                    if isinstance(attr_val, (list, tuple, np.ndarray)):
                        # For array-like attributes, convert to tuple for grakel
                        if len(attr_val) > 1:
                            # Use tuple of values for multi-dimensional attributes
                            node_attrs[i] = tuple(float(x) for x in attr_val)
                        else:
                            node_attrs[i] = float(attr_val[0])
                    else:
                        node_attrs[i] = float(attr_val)
                else:
                    # Default attribute if none exists
                    node_attrs[i] = 0.0
            else:
                node_attrs[i] = 0.0

        # Create grakel-compatible format - always include node attributes
        G_grakel.append([edges, node_attrs])

    y_grakel = y_local

    print(
        f"Dataset loaded: {len(G_grakel)} graphs, {len(np.unique(y_grakel))} classes"
    )

    return G_grakel, y_grakel, X_local, y_local

In [3]:
def main():
    """
    Main evaluation function using nested cross-validation.
    """
    print("=" * 60)
    print("GRAPH CLASSIFICATION EVALUATION - NESTED CROSS-VALIDATION")
    print("=" * 60)

    # Create log directory
    log_dir = "nested_cv_results"
    try:
        if not os.path.exists(log_dir):
            os.makedirs(log_dir)
    except OSError as e:
        print(f"Error creating log directory: {e}")
        return

    # Datasets to evaluate
    datasets_to_eval = ['bzr', 'cox2']

    # Cross-validation parameters
    n_outer_folds = 10
    n_inner_folds = 10

    all_results = {}

    for dataset_name in datasets_to_eval:
        print(f"\n{'=' * 20} {dataset_name.upper()} DATASET {'=' * 20}")

        try:
            # Load datasets
            G_grakel, y_grakel, X_local, y_local = load_dataset_from_files(
                dataset_name)

            # Evaluate graph kernel methods using nested CV
            kernel_results = evaluate_graph_kernels_nested_cv(
                dataset_name,
                G_grakel,
                y_grakel,
                log_dir,
                n_folds=n_outer_folds,
                n_inner=n_inner_folds)

            # Evaluate FGW method using nested CV
            fgw_results = evaluate_fgw_nested_cv(dataset_name,
                                                 X_local,
                                                 y_local,
                                                 log_dir,
                                                 n_folds=n_outer_folds,
                                                 n_inner=n_inner_folds)

            # Store results
            all_results[dataset_name] = {
                'kernel_methods': kernel_results,
                'fgw_method': fgw_results
            }

        except Exception as e:
            print(f"Error evaluating {dataset_name}: {e}")
            all_results[dataset_name] = None

    # Print summary
    print(f"\n{'=' * 20} SUMMARY {'=' * 20}")
    print(
        f"Nested cross-validation completed with {n_outer_folds} outer folds and {n_inner_folds} inner folds."
    )
    print(f"Results are saved in log files in the '{log_dir}' directory.")

    for dataset_name, results in all_results.items():
        if results is not None:
            print(f"\n{dataset_name.upper()} Results:")

            if results['kernel_methods']:
                for method, status in results['kernel_methods'].items():
                    print(f"  {method}: {status}")

            print(f"  FGW: {results['fgw_method']}")
        else:
            print(f"\n{dataset_name.upper()}: Evaluation failed")

    print(
        f"\nDetailed results and statistics can be found in the log files in '{log_dir}/'"
    )


if __name__ == "__main__":
    main()

GRAPH CLASSIFICATION EVALUATION - NESTED CROSS-VALIDATION

==================== BZR DATASET ====================
Loading bzr dataset...
Dataset loaded: 405 graphs, 2 classes

Evaluating graph kernel methods on bzr dataset using nested CV...

--- Evaluating hopper method with nested CV ---
explode_tuned_parameters input: [{'C': [1e-05, 5.1794746792312125e-05, 0.0002682695795279727, 0.0013894954943731374, 0.0071968567300115215, 0.037275937203149416, 0.19306977288832497, 1.0, 5.1794746792312125, 26.826957952797272, 138.9495494373139, 719.6856730011528, 3727.593720314938, 19306.977288832535, 100000.0], 'kernel_params': [[{'name': 'graph_hopper', 'kernel_type': 'linear'}]], 'normalize': [True]}]
tuned_parameters2: {'C': [1e-05, 5.1794746792312125e-05, 0.0002682695795279727, 0.0013894954943731374, 0.0071968567300115215, 0.037275937203149416, 0.19306977288832497, 1.0, 5.1794746792312125, 26.826957952797272, 138.9495494373139, 719.6856730011528, 3727.593720314938, 19306.977288832535, 100000.0]

KeyboardInterrupt: 

In [4]:
## PROTEINS ##
def evaluate_fgw_nested_cv_new(dataset_name,
                           X,
                           y,
                           log_dir,
                           n_folds=10,
                           n_inner=10,
                           random_state=42):
    """
    Evaluate FGW method using nested cross-validation.

    Parameters
    ----------
    dataset_name : str
        Name of the dataset being evaluated
    X : array
        Graph objects
    y : array
        Labels
    log_dir : str
        Directory for log files
    n_folds : int
        Number of outer cross-validation folds
    n_inner : int
        Number of inner cross-validation folds
    random_state : int
        Random state for reproducibility
    """
    print(
        f"\n--- Evaluating FGW method on {dataset_name} dataset with nested CV ---"
    )

    # Setup logger for FGW
    logger_name = f'fgw_{dataset_name}'
    logger = setup_logger(logger_name,
                          f'{log_dir}/{logger_name}_nested_cv.log')

    # Parameter grid for FGW nested CV
    # Using automatic CV alpha grid
    nb = 15
    N = int(15 / 3)
    a = np.logspace(-6, -1, N)
    c = 1 - a
    b = np.array(
        list(set(np.linspace(a[0], c[0], N)).difference(set((a[0], c[0])))))
    alphalist = np.concatenate((a, b, c))
    alpha_list = list(np.sort(np.append([0, 1], alphalist)))

    tuned_parameters = [{
        'alpha': alpha_list,
        'C': list(np.logspace(-4, 4, 15)),
        'gamma': list([2**k for k in np.linspace(-10, 10, 15)]),
        'features_metric': ['sqeuclidean'],
        'method': ['shortest_path'],
        'wl': [0],
        'amijo': [False]
    }]

    try:
        # Run nested cross-validation
        # Ensure y is a 1D array
        y_flat = np.array(y).ravel()
        nested_fgw(X,
                   y_flat,
                   tuned_parameters,
                   dataset_name,
                   logger,
                   path=None,
                   n_inner=n_inner,
                   n_iter=n_folds,
                   verbose=1)

        print(f"  FGW nested CV completed - results in log file")
        return "Completed - check log file for results"

    except Exception as e:
        print(f"  Error in FGW nested CV: {e}")
        return f"Error: {e}"

print("=" * 60)
print("GRAPH CLASSIFICATION EVALUATION - NESTED CROSS-VALIDATION")
print("=" * 60)

# Create log directory
log_dir = "nested_cv_results"
try:
    if not os.path.exists(log_dir):
        os.makedirs(log_dir)
except OSError as e:
    print(f"Error creating log directory: {e}")

# Datasets to evaluate
datasets = ['protein']

# Cross-validation parameters
n_outer_folds = 10
n_inner_folds = 10

all_results = {}

for dataset_name in datasets:
    print(f"\n{'=' * 20} {dataset_name.upper()} DATASET {'=' * 20}")

    try:
        # Load datasets
        G_grakel, y_grakel, X_local, y_local = load_dataset_from_files(
            dataset_name)

        # Evaluate graph kernel methods using nested CV
        kernel_results = evaluate_graph_kernels_nested_cv(
            dataset_name,
            G_grakel,
            y_grakel,
            log_dir,
            n_folds=n_outer_folds,
            n_inner=n_inner_folds)

        # Evaluate FGW method using nested CV
        fgw_results = evaluate_fgw_nested_cv_new(dataset_name,
                                             X_local,
                                             y_local,
                                             log_dir,
                                             n_folds=n_outer_folds,
                                             n_inner=n_inner_folds)

        # Store results
        all_results[dataset_name] = {
            'kernel_methods': kernel_results,
            'fgw_method': fgw_results
        }

    except Exception as e:
        print(f"Error evaluating {dataset_name}: {e}")
        all_results[dataset_name] = None

# Print summary
print(f"\n{'=' * 20} SUMMARY {'=' * 20}")
print(
    f"Nested cross-validation completed with {n_outer_folds} outer folds and {n_inner_folds} inner folds."
)
print(f"Results are saved in log files in the '{log_dir}' directory.")

for dataset_name, results in all_results.items():
    if results is not None:
        print(f"\n{dataset_name.upper()} Results:")

        if results['kernel_methods']:
            for method, status in results['kernel_methods'].items():
                print(f"  {method}: {status}")

        print(f"  FGW: {results['fgw_method']}")
    else:
        print(f"\n{dataset_name.upper()}: Evaluation failed")

print(
    f"\nDetailed results and statistics can be found in the log files in '{log_dir}/'"
)

GRAPH CLASSIFICATION EVALUATION - NESTED CROSS-VALIDATION

==================== PROTEIN DATASET ====================
Loading protein dataset...
Dataset loaded: 1113 graphs, 2 classes

Evaluating graph kernel methods on protein dataset using nested CV...

--- Evaluating hopper method with nested CV ---
explode_tuned_parameters input: [{'C': [1e-05, 5.1794746792312125e-05, 0.0002682695795279727, 0.0013894954943731374, 0.0071968567300115215, 0.037275937203149416, 0.19306977288832497, 1.0, 5.1794746792312125, 26.826957952797272, 138.9495494373139, 719.6856730011528, 3727.593720314938, 19306.977288832535, 100000.0], 'kernel_params': [[{'name': 'graph_hopper', 'kernel_type': 'linear'}]], 'normalize': [True]}]
tuned_parameters2: {'C': [1e-05, 5.1794746792312125e-05, 0.0002682695795279727, 0.0013894954943731374, 0.0071968567300115215, 0.037275937203149416, 0.19306977288832497, 1.0, 5.1794746792312125, 26.826957952797272, 138.9495494373139, 719.6856730011528, 3727.593720314938, 19306.9772888325

/Users/kait/Documents/GitHub/gwgraphs/Vayer Methods/utils.py:148: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return np.array(x_train),np.array(y_train),np.array(idx_train),np.array(x_test),np.array(y_test),np.array(idx_test)
/Users/kait/Documents/GitHub/gwgraphs/Vayer Methods/utils.py:148: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return np.array(x_train),np.array(y_train),np.array(idx_train),np.array(x_test),np.array(y_test),np.array(idx_test)
/Users/kait/Documents/GitHub/gwgraphs/Vayer Methods/utils.py:148: VisibleDeprecationWarning: Creatin

  hopper nested CV completed - results in log file

--- Evaluating propa method with nested CV ---
explode_tuned_parameters input: [{'C': [1e-05, 5.1794746792312125e-05, 0.0002682695795279727, 0.0013894954943731374, 0.0071968567300115215, 0.037275937203149416, 0.19306977288832497, 1.0, 5.1794746792312125, 26.826957952797272, 138.9495494373139, 719.6856730011528, 3727.593720314938, 19306.977288832535, 100000.0], 'kernel_params': [[{'name': 'propagation', 't_max': 1, 'w': 0.0001, 'M': 'TV'}], [{'name': 'propagation', 't_max': 3, 'w': 0.0001, 'M': 'TV'}], [{'name': 'propagation', 't_max': 5, 'w': 0.0001, 'M': 'TV'}], [{'name': 'propagation', 't_max': 8, 'w': 0.0001, 'M': 'TV'}], [{'name': 'propagation', 't_max': 10, 'w': 0.0001, 'M': 'TV'}], [{'name': 'propagation', 't_max': 15, 'w': 0.0001, 'M': 'TV'}], [{'name': 'propagation', 't_max': 20, 'w': 0.0001, 'M': 'TV'}]], 'normalize': [True]}]
tuned_parameters2: {'C': [1e-05, 5.1794746792312125e-05, 0.0002682695795279727, 0.001389495494373137

/Users/kait/Documents/GitHub/gwgraphs/Vayer Methods/utils.py:148: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return np.array(x_train),np.array(y_train),np.array(idx_train),np.array(x_test),np.array(y_test),np.array(idx_test)
/Users/kait/Documents/GitHub/gwgraphs/Vayer Methods/utils.py:148: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return np.array(x_train),np.array(y_train),np.array(idx_train),np.array(x_test),np.array(y_test),np.array(idx_test)
/Users/kait/Documents/GitHub/gwgraphs/Vayer Methods/utils.py:148: VisibleDeprecationWarning: Creatin

  propa nested CV completed - results in log file

--- Evaluating FGW method on protein dataset with nested CV ---
  FGW nested CV completed - results in log file

==================== SUMMARY ====================
Nested cross-validation completed with 10 outer folds and 10 inner folds.
Results are saved in log files in the 'nested_cv_results' directory.

PROTEIN Results:
  hopper: Completed - check log file for results
  propa: Completed - check log file for results
  FGW: Completed - check log file for results

Detailed results and statistics can be found in the log files in 'nested_cv_results/'


In [5]:
"""
Driver script for evaluating GK and SPK methods on IMDb-B and IMDb-M datasets.
These datasets do not have node attributes.
"""

def evaluate_graph_kernels_imdb(dataset_name, G, y, log_dir, n_folds=10, n_inner=10):
    """
    Evaluate GK and SPK graph kernel methods on IMDb datasets using nested cross-validation.
    
    Parameters
    ----------
    dataset_name : str
        Name of the dataset being evaluated ('imdb-b' or 'imdb-m')
    G : array
        Graph data in grakel format
    y : array
        Labels
    log_dir : str
        Directory for log files
    n_folds : int
        Number of outer cross-validation folds
    n_inner : int
        Number of inner cross-validation folds
    """
    results = {}
    
    # Define parameters for GK (Graphlet Sampling Kernel)
    gkklist = [3, 4, 5]
    S = [{"delta": 0.1, "epsilon": 0.1}, 
         {"delta": 0.05, "epsilon": 0.1}, 
         {"delta": 0.05, "epsilon": 0.05}, 
         {"delta": 0.1, "epsilon": 0.05}]
    
    # Define parameters for SPK (Shortest Path Kernel)
    # Note: IMDb datasets don't have node labels, so with_labels=False
    methods = {
        'gk': [{
            'C': list(np.logspace(-5, 5, 15)),
            'kernel_params': [[{"name": "graphlet_sampling", "k": kx, "sampling": sx}] 
                             for kx, sx in itertools.product(gkklist, S)],
            'normalize': [True]
        }],
        'spk': [{
            'C': list(np.logspace(-5, 5, 15)),
            'kernel_params': [[{"name": "shortest_path", "with_labels": False}]],
            'normalize': [True]
        }]
    }
    
    print(f"\nEvaluating graph kernel methods on {dataset_name} dataset using nested CV...")
    
    for method_name, param_grid in methods.items():
        print(f"\n--- Evaluating {method_name.upper()} method with nested CV ---")
        
        # Setup logger for this method
        logger_name = f'{method_name}_{dataset_name}'
        logger = setup_logger(logger_name, f'{log_dir}/{logger_name}_nested_cv.log')
        
        try:
            # Run nested cross-validation
            y_flat = np.array(y).ravel()
            nested_cv(G, y_flat, param_grid, logger, 
                     n_iter=n_folds, n_inner=n_inner, verbose=1)
            
            results[method_name] = "Completed - check log file for results"
            print(f"  {method_name.upper()} nested CV completed - results in log file")
            
        except Exception as e:
            print(f"  Error in {method_name.upper()} nested CV: {e}")
            results[method_name] = f"Error: {e}"
    
    return results


def load_imdb_dataset(dataset_name):
    """
    Load IMDb-B or IMDb-M dataset from local files.
    
    Parameters
    ----------
    dataset_name : str
        Either 'imdb-b' or 'imdb-m'
    
    Returns
    -------
    tuple : (graphs, labels) in grakel format
    """
    print(f"Loading {dataset_name} dataset...")
    
    # Load using local data loader without any node attributes
    X_local, y_local = load_local_data('.', dataset_name.lower(), 
                                       attributes=False, use_node_deg=False)
    
    # Convert to grakel format
    G_grakel = []
    for graph in X_local:
        nx_graph = graph.nx_graph
        nodes = sorted(list(nx_graph.nodes()))
        node_mapping = {node: i for i, node in enumerate(nodes)}
        
        # Convert edges to use integer indices
        edges = []
        for u, v in nx_graph.edges():
            if u in node_mapping and v in node_mapping:
                u_idx = node_mapping[u]
                v_idx = node_mapping[v]
                edges.append((u_idx, v_idx))
        
        # Create grakel-compatible format (no node attributes for IMDb datasets)
        G_grakel.append([edges, {}])
    
    y_grakel = y_local
    
    print(f"Dataset loaded: {len(G_grakel)} graphs, {len(np.unique(y_grakel))} classes")
    
    return G_grakel, y_grakel


def main():
    """
    Main evaluation function for GK and SPK on IMDb datasets.
    """
    print("=" * 60)
    print("GRAPH CLASSIFICATION EVALUATION - IMDb DATASETS")
    print("GK (Graphlet Kernel) and SPK (Shortest Path Kernel)")
    print("=" * 60)
    
    # Create log directory
    log_dir = "imdb_results"
    try:
        if not os.path.exists(log_dir):
            os.makedirs(log_dir)
    except OSError as e:
        print(f"Error creating log directory: {e}")
        return
    
    # Datasets to evaluate
    datasets_to_eval = ['imdb-b', 'imdb-m']
    
    # Cross-validation parameters
    n_outer_folds = 10
    n_inner_folds = 10
    
    all_results = {}
    
    for dataset_name in datasets_to_eval:
        print(f"\n{'=' * 20} {dataset_name.upper()} DATASET {'=' * 20}")
        
        try:
            # Load dataset
            G_grakel, y_grakel = load_imdb_dataset(dataset_name)
            
            # Evaluate graph kernel methods using nested CV
            kernel_results = evaluate_graph_kernels_imdb(
                dataset_name, G_grakel, y_grakel, log_dir,
                n_folds=n_outer_folds, n_inner=n_inner_folds
            )
            
            # Store results
            all_results[dataset_name] = kernel_results
            
        except Exception as e:
            print(f"Error evaluating {dataset_name}: {e}")
            all_results[dataset_name] = None
    
    # Print summary
    print(f"\n{'=' * 20} SUMMARY {'=' * 20}")
    print(f"Nested cross-validation completed with {n_outer_folds} outer folds and {n_inner_folds} inner folds.")
    print(f"Results are saved in log files in the '{log_dir}' directory.")
    
    for dataset_name, results in all_results.items():
        if results is not None:
            print(f"\n{dataset_name.upper()} Results:")
            for method, status in results.items():
                print(f"  {method.upper()}: {status}")
        else:
            print(f"\n{dataset_name.upper()}: Evaluation failed")
    
    print(f"\nDetailed results and statistics can be found in '{log_dir}/'")

if __name__ == "__main__":
    main()

GRAPH CLASSIFICATION EVALUATION - IMDb DATASETS
GK (Graphlet Kernel) and SPK (Shortest Path Kernel)

==================== IMDB-B DATASET ====================
Loading imdb-b dataset...
Dataset loaded: 1000 graphs, 2 classes

Evaluating graph kernel methods on imdb-b dataset using nested CV...

--- Evaluating GK method with nested CV ---
explode_tuned_parameters input: [{'C': [1e-05, 5.1794746792312125e-05, 0.0002682695795279727, 0.0013894954943731374, 0.0071968567300115215, 0.037275937203149416, 0.19306977288832497, 1.0, 5.1794746792312125, 26.826957952797272, 138.9495494373139, 719.6856730011528, 3727.593720314938, 19306.977288832535, 100000.0], 'kernel_params': [[{'name': 'graphlet_sampling', 'k': 3, 'sampling': {'delta': 0.1, 'epsilon': 0.1}}], [{'name': 'graphlet_sampling', 'k': 3, 'sampling': {'delta': 0.05, 'epsilon': 0.1}}], [{'name': 'graphlet_sampling', 'k': 3, 'sampling': {'delta': 0.05, 'epsilon': 0.05}}], [{'name': 'graphlet_sampling', 'k': 3, 'sampling': {'delta': 0.1, 'eps

/Users/kait/Documents/GitHub/gwgraphs/Vayer Methods/utils.py:148: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return np.array(x_train),np.array(y_train),np.array(idx_train),np.array(x_test),np.array(y_test),np.array(idx_test)
/Users/kait/Documents/GitHub/gwgraphs/Vayer Methods/utils.py:148: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return np.array(x_train),np.array(y_train),np.array(idx_train),np.array(x_test),np.array(y_test),np.array(idx_test)
/Users/kait/Documents/GitHub/gwgraphs/Vayer Methods/utils.py:148: VisibleDeprecationWarning: Creatin

  GK nested CV completed - results in log file

--- Evaluating SPK method with nested CV ---
explode_tuned_parameters input: [{'C': [1e-05, 5.1794746792312125e-05, 0.0002682695795279727, 0.0013894954943731374, 0.0071968567300115215, 0.037275937203149416, 0.19306977288832497, 1.0, 5.1794746792312125, 26.826957952797272, 138.9495494373139, 719.6856730011528, 3727.593720314938, 19306.977288832535, 100000.0], 'kernel_params': [[{'name': 'shortest_path', 'with_labels': False}]], 'normalize': [True]}]
tuned_parameters2: {'C': [1e-05, 5.1794746792312125e-05, 0.0002682695795279727, 0.0013894954943731374, 0.0071968567300115215, 0.037275937203149416, 0.19306977288832497, 1.0, 5.1794746792312125, 26.826957952797272, 138.9495494373139, 719.6856730011528, 3727.593720314938, 19306.977288832535, 100000.0], 'kernel_params': [[{'name': 'shortest_path', 'with_labels': False}]], 'normalize': [True]}
varNames: ['C', 'kernel_params', 'normalize']
First few combinations: [{'C': 1e-05, 'kernel_params': [{'na

/Users/kait/Documents/GitHub/gwgraphs/Vayer Methods/utils.py:148: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return np.array(x_train),np.array(y_train),np.array(idx_train),np.array(x_test),np.array(y_test),np.array(idx_test)
/Users/kait/Documents/GitHub/gwgraphs/Vayer Methods/utils.py:148: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return np.array(x_train),np.array(y_train),np.array(idx_train),np.array(x_test),np.array(y_test),np.array(idx_test)
/Users/kait/Documents/GitHub/gwgraphs/Vayer Methods/utils.py:148: VisibleDeprecationWarning: Creatin

  SPK nested CV completed - results in log file

==================== IMDB-M DATASET ====================
Loading imdb-m dataset...
Dataset loaded: 1500 graphs, 3 classes

Evaluating graph kernel methods on imdb-m dataset using nested CV...

--- Evaluating GK method with nested CV ---
explode_tuned_parameters input: [{'C': [1e-05, 5.1794746792312125e-05, 0.0002682695795279727, 0.0013894954943731374, 0.0071968567300115215, 0.037275937203149416, 0.19306977288832497, 1.0, 5.1794746792312125, 26.826957952797272, 138.9495494373139, 719.6856730011528, 3727.593720314938, 19306.977288832535, 100000.0], 'kernel_params': [[{'name': 'graphlet_sampling', 'k': 3, 'sampling': {'delta': 0.1, 'epsilon': 0.1}}], [{'name': 'graphlet_sampling', 'k': 3, 'sampling': {'delta': 0.05, 'epsilon': 0.1}}], [{'name': 'graphlet_sampling', 'k': 3, 'sampling': {'delta': 0.05, 'epsilon': 0.05}}], [{'name': 'graphlet_sampling', 'k': 3, 'sampling': {'delta': 0.1, 'epsilon': 0.05}}], [{'name': 'graphlet_sampling', 'k': 

/Users/kait/Documents/GitHub/gwgraphs/Vayer Methods/utils.py:148: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return np.array(x_train),np.array(y_train),np.array(idx_train),np.array(x_test),np.array(y_test),np.array(idx_test)
/Users/kait/Documents/GitHub/gwgraphs/Vayer Methods/utils.py:148: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return np.array(x_train),np.array(y_train),np.array(idx_train),np.array(x_test),np.array(y_test),np.array(idx_test)
/Users/kait/Documents/GitHub/gwgraphs/Vayer Methods/utils.py:148: VisibleDeprecationWarning: Creatin

  GK nested CV completed - results in log file

--- Evaluating SPK method with nested CV ---
explode_tuned_parameters input: [{'C': [1e-05, 5.1794746792312125e-05, 0.0002682695795279727, 0.0013894954943731374, 0.0071968567300115215, 0.037275937203149416, 0.19306977288832497, 1.0, 5.1794746792312125, 26.826957952797272, 138.9495494373139, 719.6856730011528, 3727.593720314938, 19306.977288832535, 100000.0], 'kernel_params': [[{'name': 'shortest_path', 'with_labels': False}]], 'normalize': [True]}]
tuned_parameters2: {'C': [1e-05, 5.1794746792312125e-05, 0.0002682695795279727, 0.0013894954943731374, 0.0071968567300115215, 0.037275937203149416, 0.19306977288832497, 1.0, 5.1794746792312125, 26.826957952797272, 138.9495494373139, 719.6856730011528, 3727.593720314938, 19306.977288832535, 100000.0], 'kernel_params': [[{'name': 'shortest_path', 'with_labels': False}]], 'normalize': [True]}
varNames: ['C', 'kernel_params', 'normalize']
First few combinations: [{'C': 1e-05, 'kernel_params': [{'na

/Users/kait/Documents/GitHub/gwgraphs/Vayer Methods/utils.py:148: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return np.array(x_train),np.array(y_train),np.array(idx_train),np.array(x_test),np.array(y_test),np.array(idx_test)
/Users/kait/Documents/GitHub/gwgraphs/Vayer Methods/utils.py:148: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return np.array(x_train),np.array(y_train),np.array(idx_train),np.array(x_test),np.array(y_test),np.array(idx_test)
/Users/kait/Documents/GitHub/gwgraphs/Vayer Methods/utils.py:148: VisibleDeprecationWarning: Creatin

  SPK nested CV completed - results in log file

==================== SUMMARY ====================
Nested cross-validation completed with 10 outer folds and 10 inner folds.
Results are saved in log files in the 'imdb_results' directory.

IMDB-B Results:
  GK: Completed - check log file for results
  SPK: Completed - check log file for results

IMDB-M Results:
  GK: Completed - check log file for results
  SPK: Completed - check log file for results

Detailed results and statistics can be found in 'imdb_results/'


In [5]:
def convert_nx_to_grakel_format(nx_graphs_with_labels):
    """
    Convert NetworkX graphs to grakel format (edge list without node attributes).
    
    Parameters
    ----------
    nx_graphs_with_labels : list of tuples
        List of (networkx_graph, label) tuples
    
    Returns
    -------
    G_grakel : list
        Graphs in grakel format
    y : array
        Labels
    """
    G_grakel = []
    y = []
    
    for nx_graph, label in nx_graphs_with_labels:
        # Get sorted node list for consistent indexing
        nodes = sorted(list(nx_graph.nodes()))
        node_mapping = {node: i for i, node in enumerate(nodes)}
        
        # Convert edges to integer indices
        edges = []
        for u, v in nx_graph.edges():
            u_idx = node_mapping[u]
            v_idx = node_mapping[v]
            edges.append((u_idx, v_idx))
        
        # Grakel format: [edges, node_labels_dict]
        # Empty dict {} means no node attributes
        G_grakel.append([edges, {}])
        y.append(label)
    
    return G_grakel, np.array(y)

In [6]:
# Generate ERSBM1
prob_matrix = [
    [0.15, 0.05],  # Probabilities related to block 1 connections
    [0.05, 0.15]  # Probabilities related to block 2 connections
]

graphs_er = generate_erdos_renyi_graphs(100, 100, 0.1)
graphs_sbm = generate_sbm_graphs(100, [50, 50], prob_matrix)

full_graph_data = graphs_er + graphs_sbm
distance_matrices = [
    compute_distance_matrix(graph) for graph, label in full_graph_data
]
labels = [label for graph, label in full_graph_data]

# Convert to grakel format for GK/SPK methods
G_grakel_ersbm1, y_grakel_ersbm1 = convert_nx_to_grakel_format(full_graph_data)

# Generate ERSBM2
prob_matrix5 = [
    [0.45, 0.05],  # Probabilities related to block 1 connections
    [0.05, 0.45]  # Probabilities related to block 2 connections
]

graphs_er5 = generate_erdos_renyi_graphs(100, 100, 0.25)
graphs_sbm5 = generate_sbm_graphs(100, [50, 50], prob_matrix5)

full_graph_data5 = graphs_er5 + graphs_sbm5
distance_matrices5 = [
    compute_distance_matrix(graph) for graph, label in full_graph_data5
]
labels5 = [label for graph, label in full_graph_data5]

# Convert to grakel format for GK/SPK methods
G_grakel_ersbm2, y_grakel_ersbm2 = convert_nx_to_grakel_format(full_graph_data5)

In [8]:
def main():
    """
    Main evaluation function using nested cross-validation.
    """
    print("=" * 60)
    print("GRAPH CLASSIFICATION EVALUATION - NESTED CROSS-VALIDATION")
    print("=" * 60)

    # Create log directory
    log_dir = "nested_cv_results"
    try:
        if not os.path.exists(log_dir):
            os.makedirs(log_dir)
    except OSError as e:
        print(f"Error creating log directory: {e}")
        return
    
    # Synthetic datasets
    synthetic_datasets = {
        'ersbm1': (G_grakel_ersbm1, y_grakel_ersbm1),
        'ersbm2': (G_grakel_ersbm2, y_grakel_ersbm2)
    }

    # Cross-validation parameters
    n_outer_folds = 10
    n_inner_folds = 10

    all_results = {}
    
    for synth_name, (G_synth, y_synth) in synthetic_datasets.items():
        print(f"\n{'=' * 20} {synth_name.upper()} DATASET {'=' * 20}")
        
        try:
            # Define parameters for GK and SPK (no node labels/attributes)
            gkklist = [3, 4, 5]
            S = [{"delta": 0.1, "epsilon": 0.1}, 
                 {"delta": 0.05, "epsilon": 0.1}, 
                 {"delta": 0.05, "epsilon": 0.05}, 
                 {"delta": 0.1, "epsilon": 0.05}]
            
            methods_synth = {
                'gk': [{
                    'C': list(np.logspace(-5, 5, 15)),
                    'kernel_params': [[{"name": "graphlet_sampling", "k": kx, "sampling": sx}] 
                                     for kx, sx in itertools.product(gkklist, S)],
                    'normalize': [True]
                }],
                'spk': [{
                    'C': list(np.logspace(-5, 5, 15)),
                    'kernel_params': [[{"name": "shortest_path", "with_labels": False}]],
                    'normalize': [True]
                }]
            }
            
            synth_results = {}
            for method_name, param_grid in methods_synth.items():
                print(f"\n--- Evaluating {method_name} method on {synth_name} ---")
                
                logger_name = f'{method_name}_{synth_name}'
                logger = setup_logger(logger_name, f'{log_dir}/{logger_name}_nested_cv.log')
                
                try:
                    nested_cv(G_synth, y_synth, param_grid, logger, 
                             n_iter=n_outer_folds, n_inner=n_inner_folds, verbose=1)
                    synth_results[method_name] = "Completed - check log file for results"
                    print(f"  {method_name} nested CV completed")
                except Exception as e:
                    print(f"  Error in {method_name}: {e}")
                    synth_results[method_name] = f"Error: {e}"
            
            all_results[synth_name] = {'kernel_methods': synth_results}
            
        except Exception as e:
            print(f"Error evaluating {synth_name}: {e}")
            all_results[synth_name] = None
    
    # Print summary
    print(f"\n{'=' * 20} SUMMARY {'=' * 20}")
    print(
        f"Nested cross-validation completed with {n_outer_folds} outer folds and {n_inner_folds} inner folds."
    )
    print(f"Results are saved in log files in the '{log_dir}' directory.")

    for dataset_name, results in all_results.items():
        if results is not None:
            print(f"\n{dataset_name.upper()} Results:")

            if 'kernel_methods' in results and results['kernel_methods']:
                for method, status in results['kernel_methods'].items():
                    print(f"  {method}: {status}")

            if 'fgw_method' in results:
                print(f"  FGW: {results['fgw_method']}")
        else:
            print(f"\n{dataset_name.upper()}: Evaluation failed")

    print(f"\nDetailed results and statistics can be found in the log files in '{log_dir}/'")
        

if __name__ == "__main__":
    main()

GRAPH CLASSIFICATION EVALUATION - NESTED CROSS-VALIDATION

==================== ERSBM1 DATASET ====================

--- Evaluating gk method on ersbm1 ---
explode_tuned_parameters input: [{'C': [1e-05, 5.1794746792312125e-05, 0.0002682695795279727, 0.0013894954943731374, 0.0071968567300115215, 0.037275937203149416, 0.19306977288832497, 1.0, 5.1794746792312125, 26.826957952797272, 138.9495494373139, 719.6856730011528, 3727.593720314938, 19306.977288832535, 100000.0], 'kernel_params': [[{'name': 'graphlet_sampling', 'k': 3, 'sampling': {'delta': 0.1, 'epsilon': 0.1}}], [{'name': 'graphlet_sampling', 'k': 3, 'sampling': {'delta': 0.05, 'epsilon': 0.1}}], [{'name': 'graphlet_sampling', 'k': 3, 'sampling': {'delta': 0.05, 'epsilon': 0.05}}], [{'name': 'graphlet_sampling', 'k': 3, 'sampling': {'delta': 0.1, 'epsilon': 0.05}}], [{'name': 'graphlet_sampling', 'k': 4, 'sampling': {'delta': 0.1, 'epsilon': 0.1}}], [{'name': 'graphlet_sampling', 'k': 4, 'sampling': {'delta': 0.05, 'epsilon': 0.1

/Users/kait/Documents/GitHub/gwgraphs/Vayer Methods/utils.py:148: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return np.array(x_train),np.array(y_train),np.array(idx_train),np.array(x_test),np.array(y_test),np.array(idx_test)
/Users/kait/Documents/GitHub/gwgraphs/Vayer Methods/utils.py:148: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return np.array(x_train),np.array(y_train),np.array(idx_train),np.array(x_test),np.array(y_test),np.array(idx_test)
/Users/kait/Documents/GitHub/gwgraphs/Vayer Methods/utils.py:148: VisibleDeprecationWarning: Creatin

  gk nested CV completed

--- Evaluating spk method on ersbm1 ---
explode_tuned_parameters input: [{'C': [1e-05, 5.1794746792312125e-05, 0.0002682695795279727, 0.0013894954943731374, 0.0071968567300115215, 0.037275937203149416, 0.19306977288832497, 1.0, 5.1794746792312125, 26.826957952797272, 138.9495494373139, 719.6856730011528, 3727.593720314938, 19306.977288832535, 100000.0], 'kernel_params': [[{'name': 'shortest_path', 'with_labels': False}]], 'normalize': [True]}]
tuned_parameters2: {'C': [1e-05, 5.1794746792312125e-05, 0.0002682695795279727, 0.0013894954943731374, 0.0071968567300115215, 0.037275937203149416, 0.19306977288832497, 1.0, 5.1794746792312125, 26.826957952797272, 138.9495494373139, 719.6856730011528, 3727.593720314938, 19306.977288832535, 100000.0], 'kernel_params': [[{'name': 'shortest_path', 'with_labels': False}]], 'normalize': [True]}
varNames: ['C', 'kernel_params', 'normalize']
First few combinations: [{'C': 1e-05, 'kernel_params': [{'name': 'shortest_path', 'with

/Users/kait/Documents/GitHub/gwgraphs/Vayer Methods/utils.py:148: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return np.array(x_train),np.array(y_train),np.array(idx_train),np.array(x_test),np.array(y_test),np.array(idx_test)
/Users/kait/Documents/GitHub/gwgraphs/Vayer Methods/utils.py:148: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return np.array(x_train),np.array(y_train),np.array(idx_train),np.array(x_test),np.array(y_test),np.array(idx_test)
/Users/kait/Documents/GitHub/gwgraphs/Vayer Methods/utils.py:148: VisibleDeprecationWarning: Creatin

  spk nested CV completed

==================== ERSBM2 DATASET ====================

--- Evaluating gk method on ersbm2 ---
explode_tuned_parameters input: [{'C': [1e-05, 5.1794746792312125e-05, 0.0002682695795279727, 0.0013894954943731374, 0.0071968567300115215, 0.037275937203149416, 0.19306977288832497, 1.0, 5.1794746792312125, 26.826957952797272, 138.9495494373139, 719.6856730011528, 3727.593720314938, 19306.977288832535, 100000.0], 'kernel_params': [[{'name': 'graphlet_sampling', 'k': 3, 'sampling': {'delta': 0.1, 'epsilon': 0.1}}], [{'name': 'graphlet_sampling', 'k': 3, 'sampling': {'delta': 0.05, 'epsilon': 0.1}}], [{'name': 'graphlet_sampling', 'k': 3, 'sampling': {'delta': 0.05, 'epsilon': 0.05}}], [{'name': 'graphlet_sampling', 'k': 3, 'sampling': {'delta': 0.1, 'epsilon': 0.05}}], [{'name': 'graphlet_sampling', 'k': 4, 'sampling': {'delta': 0.1, 'epsilon': 0.1}}], [{'name': 'graphlet_sampling', 'k': 4, 'sampling': {'delta': 0.05, 'epsilon': 0.1}}], [{'name': 'graphlet_samplin

/Users/kait/Documents/GitHub/gwgraphs/Vayer Methods/utils.py:148: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return np.array(x_train),np.array(y_train),np.array(idx_train),np.array(x_test),np.array(y_test),np.array(idx_test)
/Users/kait/Documents/GitHub/gwgraphs/Vayer Methods/utils.py:148: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return np.array(x_train),np.array(y_train),np.array(idx_train),np.array(x_test),np.array(y_test),np.array(idx_test)
/Users/kait/Documents/GitHub/gwgraphs/Vayer Methods/utils.py:148: VisibleDeprecationWarning: Creatin

  gk nested CV completed

--- Evaluating spk method on ersbm2 ---
explode_tuned_parameters input: [{'C': [1e-05, 5.1794746792312125e-05, 0.0002682695795279727, 0.0013894954943731374, 0.0071968567300115215, 0.037275937203149416, 0.19306977288832497, 1.0, 5.1794746792312125, 26.826957952797272, 138.9495494373139, 719.6856730011528, 3727.593720314938, 19306.977288832535, 100000.0], 'kernel_params': [[{'name': 'shortest_path', 'with_labels': False}]], 'normalize': [True]}]
tuned_parameters2: {'C': [1e-05, 5.1794746792312125e-05, 0.0002682695795279727, 0.0013894954943731374, 0.0071968567300115215, 0.037275937203149416, 0.19306977288832497, 1.0, 5.1794746792312125, 26.826957952797272, 138.9495494373139, 719.6856730011528, 3727.593720314938, 19306.977288832535, 100000.0], 'kernel_params': [[{'name': 'shortest_path', 'with_labels': False}]], 'normalize': [True]}
varNames: ['C', 'kernel_params', 'normalize']
First few combinations: [{'C': 1e-05, 'kernel_params': [{'name': 'shortest_path', 'with

/Users/kait/Documents/GitHub/gwgraphs/Vayer Methods/utils.py:148: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return np.array(x_train),np.array(y_train),np.array(idx_train),np.array(x_test),np.array(y_test),np.array(idx_test)
/Users/kait/Documents/GitHub/gwgraphs/Vayer Methods/utils.py:148: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return np.array(x_train),np.array(y_train),np.array(idx_train),np.array(x_test),np.array(y_test),np.array(idx_test)
/Users/kait/Documents/GitHub/gwgraphs/Vayer Methods/utils.py:148: VisibleDeprecationWarning: Creatin

  spk nested CV completed

==================== SUMMARY ====================
Nested cross-validation completed with 10 outer folds and 10 inner folds.
Results are saved in log files in the 'nested_cv_results' directory.

ERSBM1 Results:
  gk: Completed - check log file for results
  spk: Completed - check log file for results

ERSBM2 Results:
  gk: Completed - check log file for results
  spk: Completed - check log file for results

Detailed results and statistics can be found in the log files in 'nested_cv_results/'


/Users/kait/Documents/GitHub/gwgraphs/Vayer Methods/utils.py:148: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return np.array(x_train),np.array(y_train),np.array(idx_train),np.array(x_test),np.array(y_test),np.array(idx_test)
/Users/kait/Documents/GitHub/gwgraphs/Vayer Methods/utils.py:148: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return np.array(x_train),np.array(y_train),np.array(idx_train),np.array(x_test),np.array(y_test),np.array(idx_test)
